#Semana 4 Ejercicios: 
* Modelado Gold: Star, Snowflake, OBT, Galaxy
* INSERT INTO, INSERT OVERWRITE, MERGE y MD5
* Idempotencia: problema, soluciones y particiones

In [0]:
SELECT COUNT(*) FROM bootcamp.silver.propiedades;
DESCRIBE TABLE bootcamp.silver.propiedades;

In [0]:
-- dim_zona - Crear dimension zona
DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE bootcamp.gold.dim_zona (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    partido STRING NOT NULL,
    region STRING NOT NULL,
    ciudad STRING,
    provincia STRING DEFAULT 'Buenos Aires',
    pais STRING DEFAULT 'Argentina',
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de zonas - Star Schema';

INSERT INTO bootcamp.gold.dim_zona (partido, region, ciudad, provincia, pais)
SELECT DISTINCT
    sp.partido,
    sp.region,
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END as ciudad,
    'Buenos Aires' as provincia,
    'Argentina' as pais
FROM bootcamp.silver.propiedades sp
WHERE sp.partido IS NOT NULL
ORDER BY sp.partido;

In [0]:
-- dim_tipo_operacion
DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE bootcamp.gold.dim_tipo_operacion (
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    tipo_operacion STRING NOT NULL,
    moneda STRING NOT NULL,
    categoria STRING,
    descripcion STRING,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de tipo de operación + moneda - Star Schema';

INSERT INTO bootcamp.gold.dim_tipo_operacion (tipo_operacion, moneda, categoria, descripcion)
SELECT DISTINCT
    tipo_operacion,
    moneda,
    CASE 
        WHEN tipo_operacion = 'alquiler' THEN 'residencial'
        WHEN tipo_operacion = 'venta' THEN 'residencial'
        WHEN tipo_operacion = 'alquiler_temporario' THEN 'temporal'
        ELSE 'otro'
    END as categoria,
    CASE 
        WHEN tipo_operacion = 'alquiler' THEN 'Alquiler residencial'
        WHEN tipo_operacion = 'venta' THEN 'Venta de propiedad'
        WHEN tipo_operacion = 'alquiler_temporario' THEN 'Alquiler temporal'
        ELSE 'Otro tipo'
    END as descripcion
FROM bootcamp.silver.propiedades
WHERE tipo_operacion IS NOT NULL AND moneda IS NOT NULL
ORDER BY tipo_operacion, moneda;

In [0]:
-- dim_tiempo
DROP TABLE IF EXISTS bootcamp.gold.dim_tiempo;

CREATE TABLE bootcamp.gold.dim_tiempo (
    fecha_id BIGINT,
    fecha DATE NOT NULL,
    anio INT,
    mes INT,
    trimestre INT,
    dia_semana STRING,
    es_fin_de_semana BOOLEAN,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de tiempo - Star Schema';

INSERT INTO bootcamp.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)
SELECT DISTINCT
    CAST(DATE_FORMAT(fecha_publicacion, 'yyyyMMdd') AS BIGINT) as fecha_id,
    fecha_publicacion as fecha,
    YEAR(fecha_publicacion) as anio,
    MONTH(fecha_publicacion) as mes,
    QUARTER(fecha_publicacion) as trimestre,
    CASE DAYOFWEEK(fecha_publicacion)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END as dia_semana,
    DAYOFWEEK(fecha_publicacion) IN (1, 7) as es_fin_de_semana
FROM bootcamp.silver.propiedades
WHERE fecha_publicacion IS NOT NULL;

In [0]:
-- dim_caracteristicas (Kimball junk dimension — ¿Cómo es la propiedad?)
DROP TABLE IF EXISTS bootcamp.gold.dim_caracteristicas;

CREATE TABLE bootcamp.gold.dim_caracteristicas (
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    estado STRING NOT NULL,
    cochera BOOLEAN,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de características de la propiedad (estado + cochera) - Star Schema';

INSERT INTO bootcamp.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
    COALESCE(estado, 'sin especificar') as estado,
    COALESCE(cochera, false) as cochera
FROM bootcamp.silver.propiedades
ORDER BY estado, cochera;

In [0]:
-- dim_orientacion
DROP TABLE IF EXISTS bootcamp.gold.dim_orientacion;

CREATE TABLE bootcamp.gold.dim_orientacion (
    orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    orientacion STRING NOT NULL,
    tipo_orientacion STRING,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de orientación - Star Schema';

INSERT INTO bootcamp.gold.dim_orientacion (orientacion, tipo_orientacion)
SELECT DISTINCT
    COALESCE(orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(orientacion) LIKE '%este%' THEN 'este'
        WHEN LOWER(orientacion) LIKE '%oeste%' THEN 'oeste'
        ELSE 'sin especificar'
    END as tipo_orientacion
FROM bootcamp.silver.propiedades
ORDER BY orientacion;

## Ejercicio 1.2: Crear tabla de hechos (fact_propiedades)

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.fact_propiedades;

CREATE TABLE bootcamp.gold.fact_propiedades (
    row_hash STRING NOT NULL,
    zona_id BIGINT,
    tipo_operacion_id BIGINT,
    fecha_id BIGINT,
    caracteristicas_id BIGINT,
    orientacion_id BIGINT,
    precio DECIMAL(15,2),
    moneda STRING,
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)';

INSERT INTO bootcamp.gold.fact_propiedades (
    row_hash, zona_id, tipo_operacion_id, fecha_id, caracteristicas_id, orientacion_id,
    precio, expensas, precio_por_m2,
    metros_cuadrados_totales, metros_cuadrados_cubiertos, ambientes, url
)
SELECT 
    MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
    dz.zona_id,
    dt.tipo_operacion_id,
    dtf.fecha_id,
    dc.caracteristicas_id,
    do.orientacion_id,
    sp.precio, sp.expensas, sp.precio_por_m2,
    sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
    sp.ambientes, sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN bootcamp.gold.dim_tiempo dtf ON sp.fecha_publicacion = dtf.fecha
LEFT JOIN bootcamp.gold.dim_caracteristicas dc ON COALESCE(sp.estado, 'sin especificar') = dc.estado AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LIMIT 10000;

SELECT COUNT(*) AS total_fact, COUNT(DISTINCT row_hash) AS hashes_unicos
FROM bootcamp.gold.fact_propiedades;

## Ejercicio 1.3: Snowflake Schema


In [0]:

-- Nivel 1: Provincia
DROP TABLE IF EXISTS bootcamp.gold.dim_provincia_sf;

CREATE TABLE bootcamp.gold.dim_provincia_sf (
    provincia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    provincia STRING NOT NULL,
    pais STRING DEFAULT 'Argentina',
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de provincias - Snowflake Schema';

INSERT INTO bootcamp.gold.dim_provincia_sf (provincia, pais)
SELECT DISTINCT 'Buenos Aires' as provincia, 'Argentina' as pais;

-- Nivel 2: Ciudad
DROP TABLE IF EXISTS bootcamp.gold.dim_ciudad_sf;

CREATE TABLE bootcamp.gold.dim_ciudad_sf (
    ciudad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    ciudad STRING NOT NULL,
    provincia_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de ciudades - Snowflake Schema';

INSERT INTO bootcamp.gold.dim_ciudad_sf (ciudad, provincia_id)
SELECT DISTINCT
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END as ciudad,
    1 as provincia_id
FROM bootcamp.silver.propiedades sp
WHERE sp.region IS NOT NULL;

-- Nivel 3: Zona (referencia a ciudad)
DROP TABLE IF EXISTS bootcamp.gold.dim_zona_sf;

CREATE TABLE bootcamp.gold.dim_zona_sf (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    partido STRING NOT NULL,
    region STRING NOT NULL,
    ciudad_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de zonas - Snowflake Schema (normalizada)';

INSERT INTO bootcamp.gold.dim_zona_sf (partido, region, ciudad_id)
SELECT DISTINCT
    sp.partido,
    sp.region,
    dc.ciudad_id
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_ciudad_sf dc ON 
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END = dc.ciudad
WHERE sp.partido IS NOT NULL
ORDER BY sp.partido;

-- Query de ejemplo Snowflake (para que veas la complejidad de joins para solamente reconstruir una dimensión equivalena a la de STAR)
SELECT 
    dp.provincia,
    dc.ciudad,
    dz.partido, dz.region
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_zona_sf dz ON sp.partido = dz.partido AND sp.region = dz.region
JOIN bootcamp.gold.dim_ciudad_sf dc ON dz.ciudad_id = dc.ciudad_id
JOIN bootcamp.gold.dim_provincia_sf dp ON dc.provincia_id = dp.provincia_id
LIMIT 20;

## Ejercicio 1.4: OBT (One Big Table)

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.obt_propiedades_completa;

CREATE TABLE bootcamp.gold.obt_propiedades_completa (
    propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    precio DECIMAL(15,2),
    moneda STRING,
    expensas DECIMAL(15,2),
    ambientes INT,
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    antiguedad INT,
    cochera BOOLEAN,
    precio_por_m2 DECIMAL(15,2),
    partido STRING,
    region STRING,
    ciudad STRING,
    provincia STRING DEFAULT 'Buenos Aires',
    pais STRING DEFAULT 'Argentina',
    tipo_operacion STRING,
    categoria_operacion STRING,
    estado STRING,
    categoria_estado STRING,
    orientacion STRING,
    tipo_orientacion STRING,
    fecha_publicacion DATE,
    anio INT,
    mes INT,
    trimestre INT,
    dia_semana STRING,
    es_fin_de_semana BOOLEAN,
    segmento_precio STRING,
    rango_metros STRING,
    rango_ambientes STRING,
    tiene_expensas BOOLEAN,
    ratio_m2_cubiertos_totales DECIMAL(5,2),
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'One Big Table - Propiedades completas denormalizadas';

INSERT INTO bootcamp.gold.obt_propiedades_completa (
    precio, moneda, expensas, ambientes,
    metros_cuadrados_totales, metros_cuadrados_cubiertos,
    antiguedad, cochera, precio_por_m2,
    partido, region, ciudad, provincia, pais,
    tipo_operacion, categoria_operacion,
    estado, categoria_estado,
    orientacion, tipo_orientacion,
    fecha_publicacion, anio, mes, trimestre, dia_semana, es_fin_de_semana,
    segmento_precio, rango_metros, rango_ambientes,
    tiene_expensas, ratio_m2_cubiertos_totales,
    url
)
SELECT 
    sp.precio, sp.moneda, sp.expensas, sp.ambientes,
    sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
    sp.antiguedad, sp.cochera, sp.precio_por_m2,
    sp.partido,
    sp.region,
    CASE WHEN sp.region = 'capital federal' THEN 'CABA' ELSE 'GBA' END as ciudad,
    'Buenos Aires' as provincia, 'Argentina' as pais,
    sp.tipo_operacion,
    CASE 
        WHEN sp.tipo_operacion = 'alquiler' THEN 'residencial'
        WHEN sp.tipo_operacion = 'venta' THEN 'residencial'
        WHEN sp.tipo_operacion = 'alquiler_temporario' THEN 'temporal'
        ELSE 'otro'
    END as categoria_operacion,
    sp.estado,
    CASE 
        WHEN LOWER(sp.estado) LIKE '%nuevo%' THEN 'nuevo'
        WHEN LOWER(sp.estado) LIKE '%usado%' THEN 'usado'
        WHEN LOWER(sp.estado) LIKE '%remodelado%' THEN 'remodelado'
        ELSE 'sin especificar'
    END as categoria_estado,
    COALESCE(sp.orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(sp.orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(sp.orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(sp.orientacion) LIKE '%este%' THEN 'este'
        WHEN LOWER(sp.orientacion) LIKE '%oeste%' THEN 'oeste'
        ELSE 'sin especificar'
    END as tipo_orientacion,
    sp.fecha_publicacion,
    YEAR(sp.fecha_publicacion) as anio, MONTH(sp.fecha_publicacion) as mes,
    QUARTER(sp.fecha_publicacion) as trimestre,
    CASE DAYOFWEEK(sp.fecha_publicacion)
        WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Lunes' WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles' WHEN 5 THEN 'Jueves' WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END as dia_semana,
    DAYOFWEEK(sp.fecha_publicacion) IN (1, 7) as es_fin_de_semana,
    CASE 
        WHEN sp.moneda = 'ARS' AND sp.precio < 300000 THEN 'bajo'
        WHEN sp.moneda = 'ARS' AND sp.precio < 500000 THEN 'medio'
        WHEN sp.moneda = 'ARS' AND sp.precio < 800000 THEN 'alto'
        WHEN sp.moneda = 'ARS' THEN 'premium'
        WHEN sp.moneda = 'USD' AND sp.precio < 500 THEN 'bajo'
        WHEN sp.moneda = 'USD' AND sp.precio < 1000 THEN 'medio'
        WHEN sp.moneda = 'USD' AND sp.precio < 2000 THEN 'alto'
        WHEN sp.moneda = 'USD' THEN 'premium'
        ELSE 'sin clasificar'
    END as segmento_precio,
    CASE 
        WHEN sp.metros_cuadrados_totales < 50 THEN 'pequeño'
        WHEN sp.metros_cuadrados_totales < 100 THEN 'mediano'
        WHEN sp.metros_cuadrados_totales < 200 THEN 'grande'
        ELSE 'muy grande'
    END as rango_metros,
    CASE 
        WHEN sp.ambientes = 1 THEN 'mono'
        WHEN sp.ambientes BETWEEN 2 AND 3 THEN '2-3'
        WHEN sp.ambientes >= 4 THEN '4+'
        ELSE 'sin especificar'
    END as rango_ambientes,
    sp.expensas IS NOT NULL AND sp.expensas > 0 as tiene_expensas,
    ROUND(sp.metros_cuadrados_cubiertos / NULLIF(sp.metros_cuadrados_totales, 0), 2) as ratio_m2_cubiertos_totales,
    sp.url
FROM bootcamp.silver.propiedades sp
LIMIT 10000;

SELECT COUNT(*) AS total_obt FROM bootcamp.gold.obt_propiedades_completa;

## Ejercicio 1.5: Galaxy Schema

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.fact_consultas;

CREATE TABLE bootcamp.gold.fact_consultas (
    consulta_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    zona_id BIGINT,
    fecha_id BIGINT,
    propiedad_url STRING,
    tipo_consulta STRING,
    cantidad_consultas INT DEFAULT 1,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de hechos de consultas - Galaxy Schema (mockeada)';

INSERT INTO bootcamp.gold.fact_consultas (
    zona_id, fecha_id, propiedad_url, tipo_consulta, cantidad_consultas
)
SELECT 
    dz.zona_id,
    dtf.fecha_id,
    sp.url as propiedad_url,
    CASE (RANDOM() * 4)::INT
        WHEN 0 THEN 'vista'
        WHEN 1 THEN 'contacto'
        WHEN 2 THEN 'favorito'
        ELSE 'compartir'
    END as tipo_consulta,
    (RANDOM() * 10 + 1)::INT as cantidad_consultas
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
JOIN bootcamp.gold.dim_tiempo dtf ON sp.fecha_publicacion = dtf.fecha
WHERE sp.precio_por_m2 < (
    SELECT PERCENTILE_APPROX(precio_por_m2, 0.5) 
    FROM bootcamp.silver.propiedades
)
LIMIT 5000;

In [0]:
-- Preview: filas cargadas en fact_consultas (Galaxy)
SELECT * FROM bootcamp.gold.fact_consultas ORDER BY consulta_id LIMIT 20;

In [0]:
-- Análisis cruzado entre hechos
SELECT 
    dz.partido, dz.region,
    COUNT(DISTINCT fp.row_hash) as total_propiedades,
    COUNT(DISTINCT fc.consulta_id) as total_consultas,
    SUM(fc.cantidad_consultas) as cantidad_total_consultas,
    ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio,
    ROUND(SUM(fc.cantidad_consultas) * 1.0 / NULLIF(COUNT(DISTINCT fp.row_hash), 0), 2) as consultas_por_propiedad
FROM bootcamp.gold.dim_zona dz
LEFT JOIN bootcamp.gold.fact_propiedades fp ON dz.zona_id = fp.zona_id
LEFT JOIN bootcamp.gold.fact_consultas fc ON dz.zona_id = fc.zona_id
GROUP BY dz.partido, dz.region
HAVING COUNT(DISTINCT fp.row_hash) > 0
ORDER BY consultas_por_propiedad DESC
LIMIT 20;


## Ejercicio 1.6: Comparar esquemas

In [0]:
%sql
-- Star Schema (2 JOINs)
SELECT dz.partido, dz.region, dt.tipo_operacion, AVG(f.precio_por_m2) as avg_precio_m2
FROM bootcamp.gold.fact_propiedades f
LEFT JOIN bootcamp.gold.dim_zona dz ON f.zona_id = dz.zona_id
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt ON f.tipo_operacion_id = dt.tipo_operacion_id
GROUP BY dz.partido, dz.region, dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
-- Snowflake Schema (4 JOINs)
SELECT dz.partido, dz.region, dt.tipo_operacion, AVG(f.precio_por_m2) as avg_precio_m2
FROM bootcamp.gold.fact_propiedades f
LEFT JOIN bootcamp.gold.dim_zona_sf dz ON f.zona_id = dz.zona_id
LEFT JOIN bootcamp.gold.dim_ciudad_sf dc ON dz.ciudad_id = dc.ciudad_id
LEFT JOIN bootcamp.gold.dim_provincia_sf dp ON dc.provincia_id = dp.provincia_id
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt ON f.tipo_operacion_id = dt.tipo_operacion_id
GROUP BY dz.partido, dz.region, dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
-- OBT (0 JOINs)
SELECT partido, region, tipo_operacion, AVG(precio_por_m2) as avg_precio_m2
FROM bootcamp.gold.obt_propiedades_completa
GROUP BY partido, region, tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
%sql
-- Galaxy Schema (mismo que Star — fact_propiedades comparte dimensiones)
SELECT dz.partido, dz.region, dt.tipo_operacion, AVG(f.precio_por_m2) as avg_precio_m2
FROM bootcamp.gold.fact_propiedades f
LEFT JOIN bootcamp.gold.dim_zona dz ON f.zona_id = dz.zona_id
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt ON f.tipo_operacion_id = dt.tipo_operacion_id
GROUP BY dz.partido, dz.region, dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

## Ejercicio 1.7: Extensión del modelo — ¿Galaxy, Star u OBT?

**Respuesta sugerida (no hay única correcta):**

Lo más natural es pasar a **Galaxy Schema** agregando una segunda fact (`fact_consultas`) que comparta `dim_zona` y `dim_tiempo` con `fact_propiedades`.

- Las consultas y las propiedades **sí comparten dimensiones** (partido/region, tiempo) → Galaxy aprovecha eso
- Agregar las consultas al Star inflaría la fact existente con columnas no relacionadas (`cantidad_consultas` no es una métrica de la propiedad)
- Un OBT que unifique todo mezclaría granularidades distintas (una fila por propiedad vs una fila por consulta)

**¿Cuándo cambiaría de opinión?**
- Si el consumo es 100% ML/notebooks → OBT puede ser más práctico (cero JOINs)
- Si las consultas nunca se cruzan con propiedades → dos Stars independientes en vez de Galaxy

## Ejercicio 1.8: Star vs OBT — ¿Cuándo cambiar de opinión?

**Para un dashboard de precios por partido y tipo de operación:**
→ **Star Schema** — el dashboard hace JOINs específicos, las herramientas BI (Looker, Power BI) están optimizadas para Star.

**¿Cuándo cambiaría a OBT?**
- Si el data scientist necesita TODAS las columnas juntas para un modelo de ML (predicción de precios)
- Si las queries son ad-hoc y cambian constantemente (exploración libre, no dashboard fijo)
- Si el equipo no tiene experiencia con JOINs (OBT es `SELECT * WHERE...`)

**¿Cuándo NO cambiaría?**
- Si la tabla se actualiza frecuentemente (OBT es costoso de reconstruir)
- Si hay muchas dimensiones grandes (OBT explota en tamaño)

## Ejercicio 1.9: Diagrama guiado — De Star a Snowflake

**Star Schema actual:**
```
fact_propiedades
├── FK → dim_zona (partido, region, ciudad, provincia, pais)
├── FK → dim_tipo_operacion
├── FK → dim_tiempo
├── FK → dim_caracteristicas
└── FK → dim_orientacion
```

**Si agrego dim_barrio normalizando dim_zona:**
```
fact_propiedades
├── FK → dim_zona_sf (partido, region, FK → dim_ciudad_sf)
│                      └── dim_ciudad_sf (ciudad, FK → dim_provincia_sf)
│                                          └── dim_provincia_sf (provincia, pais)
├── FK → dim_tipo_operacion
├── FK → dim_tiempo
├── FK → dim_caracteristicas
└── FK → dim_orientacion
```

**Se convierte en Snowflake Schema** (parcial — solo dim_zona está normalizada).

**Trade-off:**
- Antes: 1 JOIN a dim_zona que ya tenía ciudad y provincia
- Ahora: 3 JOINs (partido → ciudad → provincia)
- Para 50 partidos con 2 ciudades → no vale la pena normalizar
- Para 500K partidos con jerarquía de 4 niveles → sí vale

## Ejercicio 1.10: Trade-offs — Star vs Snowflake según volumen y recursos

**Escenario A (50 partidos, 1 nodo, 8GB):**
→ **Star Schema**
- 50 valores únicos = la redundancia es insignificante (bytes)
- El nodo limitado sufre más con JOINs extra que con almacenamiento extra
- Mantener 1 tabla dim_zona es más simple que 3 tablas normalizadas

**Escenario B (500K partidos, 10 nodos, 640GB total):**
→ **Snowflake Schema**
- 500K partidos × (ciudad + provincia + pais repetidos) = redundancia significativa
- 10 nodos distribuyen los JOINs eficientemente con Spark
- Si una ciudad cambia de provincia, en Snowflake actualizás 1 fila en dim_provincia; en Star actualizás las 500K filas de dim_zona

**¿Databricks cambia la respuesta?**
- Sí: almacenamiento en Delta Lake es columnar y barato → la redundancia de Star duele menos
- Pero: con 500K zonas y jerarquías que cambian → Snowflake sigue ganando por mantenimiento
- Z-Ordering puede compensar parcialmente el costo de JOINs en Snowflake

## Ejercicio 1.11: Pregunta de entrevista — Star vs Snowflake

**Respuesta modelo (5-8 líneas):**

Elegiría Snowflake sobre Star cuando las dimensiones son grandes, tienen jerarquías naturales de múltiples niveles, y esas jerarquías cambian con frecuencia. La normalización de Snowflake reduce redundancia y hace que las actualizaciones sean puntuales (cambiar 1 fila vs miles). El precio es más JOINs y queries más complejas para el usuario final.

En un data warehouse relacional (PostgreSQL, Redshift), los JOINs adicionales de Snowflake pueden impactar significativamente el rendimiento — ahí Star suele ganar. Pero en un data lakehouse (Databricks/Spark), el procesamiento es distribuido, el almacenamiento es columnar, y con ZORDER los JOINs se optimizan bien. Esto hace que Snowflake sea más viable en lakehouses que en bases relacionales.

Ejemplo concreto: una dim_producto con 3 niveles (producto → subcategoría → categoría) y 1M de productos. En Star, cada producto repite su categoría y subcategoría. En Snowflake, la categoría se almacena una sola vez. Si renombrás una categoría, en Star actualizás 1M filas; en Snowflake, 1 fila.

---
# Módulo 2: INSERT INTO, INSERT OVERWRITE, hash y MERGE

## Ejercicio 2.1: INSERT INTO — Carga incremental

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_into;

CREATE TABLE bootcamp.silver.propiedades_into (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para INSERT INTO';

-- Primera carga: alquileres
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 500;

SELECT COUNT(*) as total_despues_primera_carga FROM bootcamp.silver.propiedades_into;

In [0]:
-- Segunda carga: ventas (INSERT INTO acumula)
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'venta'
LIMIT 500;

SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_into
GROUP BY tipo_operacion;

In [0]:
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades sp
WHERE tipo_operacion = 'alquiler_temporario'
    AND NOT EXISTS (
        SELECT 1 
        FROM bootcamp.silver.propiedades_into pi
        WHERE pi.url = sp.url
            AND pi.precio = sp.precio
    )
LIMIT 300;

SELECT COUNT(*) as total_registros FROM bootcamp.silver.propiedades_into;

## Ejercicio 2.3: INSERT OVERWRITE — Reprocesamiento

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_overwrite;

CREATE TABLE bootcamp.silver.propiedades_overwrite (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para INSERT OVERWRITE';

-- Primera carga
INSERT OVERWRITE bootcamp.silver.propiedades_overwrite
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 1000;

SELECT COUNT(*) as total_registros FROM bootcamp.silver.propiedades_overwrite;

In [0]:
-- Segunda carga: OVERWRITE reemplaza todo
INSERT OVERWRITE bootcamp.silver.propiedades_overwrite
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'venta'
LIMIT 1000;

-- Solo tiene 'venta' (no acumuló 'alquiler')
SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_overwrite
GROUP BY tipo_operacion;

## Ejercicio 2.4: Generar hash con MD5

In [0]:
SELECT 
    MD5('hola') as hash_simple,
    MD5('hola') as hash_repetido,
    MD5('chau') as hash_diferente;

In [0]:
SELECT 
    *,
    MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash
FROM bootcamp.silver.propiedades
LIMIT 5;

## Ejercicio 2.5: MERGE — Solo INSERT

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_insert;

CREATE TABLE bootcamp.silver.propiedades_merge_insert (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING PRIMARY KEY,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE (solo INSERT)';

INSERT INTO bootcamp.silver.propiedades_merge_insert
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_insert;

In [0]:
MERGE INTO bootcamp.silver.propiedades_merge_insert AS target
USING (
    SELECT partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion IN ('alquiler', 'venta')
    LIMIT 200
) AS source
ON target.url = source.url
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url, CURRENT_TIMESTAMP());

SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_merge_insert
GROUP BY tipo_operacion;

## Ejercicio 2.6: MERGE — UPSERT


In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_upsert;

CREATE TABLE bootcamp.silver.propiedades_merge_upsert (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE (UPSERT)';

INSERT INTO bootcamp.silver.propiedades_merge_upsert
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp,
    CURRENT_TIMESTAMP() as _updated_at
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_upsert;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW datos_actualizados AS
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url
FROM (
    SELECT partido, region, tipo_operacion,
        precio * 1.10 as precio, moneda,
        metros_cuadrados_totales,
        precio_por_m2 * 1.10 as precio_por_m2, url,
        ROW_NUMBER() OVER (PARTITION BY precio, url ORDER BY precio) as rn
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion = 'alquiler'
) a
WHERE rn = 1
LIMIT 150;

MERGE INTO bootcamp.silver.propiedades_merge_upsert AS target
USING datos_actualizados AS source
ON target.url = source.url AND target.precio = source.precio
WHEN MATCHED THEN
    UPDATE SET
        precio_por_m2 = source.precio_por_m2,
        _updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        _processing_timestamp, _updated_at)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

SELECT COUNT(*) as total_despues_upsert FROM bootcamp.silver.propiedades_merge_upsert;

## Ejercicio 2.7: MERGE con Hash — Deduplicación simplificada

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_hash;

CREATE TABLE bootcamp.silver.propiedades_merge_hash (
    row_hash STRING NOT NULL,
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con Hash';

INSERT INTO bootcamp.silver.propiedades_merge_hash
SELECT 
    MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
    partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 200;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_hash;

In [0]:
MERGE INTO bootcamp.silver.propiedades_merge_hash AS target
USING (
    SELECT 
        MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
        partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion IN ('alquiler', 'venta')
    LIMIT 400
) AS source
ON target.row_hash = source.row_hash
WHEN NOT MATCHED THEN
    INSERT (row_hash, partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)
    VALUES (source.row_hash, source.partido, source.region, source.tipo_operacion, source.precio,
        source.moneda, source.metros_cuadrados_totales, source.precio_por_m2,
        source.url, CURRENT_TIMESTAMP());

SELECT COUNT(*) as total_despues_merge, COUNT(DISTINCT row_hash) as hashes_unicos
FROM bootcamp.silver.propiedades_merge_hash;

## Ejercicio 2.8: MERGE — UPSERT + DELETE (soft delete)


In [0]:
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_delete;

CREATE TABLE bootcamp.silver.propiedades_merge_delete (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _is_deleted BOOLEAN DEFAULT false
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con soft delete';

INSERT INTO bootcamp.silver.propiedades_merge_delete
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial,
    SUM(CASE WHEN _is_deleted THEN 1 ELSE 0 END) as borrados
FROM bootcamp.silver.propiedades_merge_delete;

In [0]:
CREATE OR REPLACE TEMP VIEW datos_nuevos_delete AS
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 50;

MERGE INTO bootcamp.silver.propiedades_merge_delete AS target
USING datos_nuevos_delete AS source
ON target.url = source.url
WHEN MATCHED THEN
    UPDATE SET
        precio = source.precio,
        precio_por_m2 = source.precio_por_m2,
        _updated_at = CURRENT_TIMESTAMP(),
        _is_deleted = false
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        _processing_timestamp, _updated_at, _is_deleted)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false)
WHEN NOT MATCHED BY SOURCE THEN
    UPDATE SET _is_deleted = true, _updated_at = CURRENT_TIMESTAMP();

SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN _is_deleted THEN 1 ELSE 0 END) as soft_deleted,
    SUM(CASE WHEN NOT _is_deleted THEN 1 ELSE 0 END) as activos
FROM bootcamp.silver.propiedades_merge_delete;

---
# Módulo 3: Idempotencia

## Ejercicio 3.1: Demostrar el problema — INSERT INTO duplica

In [0]:
CREATE OR REPLACE TABLE bootcamp.bronze.ventas (
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    cliente_id STRING,
    timestamp_carga TIMESTAMP
) USING DELTA;

INSERT INTO bootcamp.bronze.ventas VALUES
    ('V001', '2024-01-15', 'Laptop', 1, 1200.00, 'C001', CURRENT_TIMESTAMP()),
    ('V002', '2024-01-15', 'Mouse', 2, 25.50, 'C001', CURRENT_TIMESTAMP()),
    ('V003', '2024-01-16', 'Teclado', 1, 75.00, 'C002', CURRENT_TIMESTAMP()),
    ('V004', '2024-01-16', 'Monitor', 1, 300.00, 'C003', CURRENT_TIMESTAMP()),
    ('V005', '2024-01-17', 'Laptop', 1, 1200.00, 'C004', CURRENT_TIMESTAMP());

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.ventas_no_idempotente;

CREATE TABLE bootcamp.silver.ventas_no_idempotente (
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;

-- Primera ejecución
INSERT INTO bootcamp.silver.ventas_no_idempotente
SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id
FROM bootcamp.bronze.ventas;

SELECT COUNT(*) as filas_primera_ejecucion FROM bootcamp.silver.ventas_no_idempotente;

In [0]:
-- Segunda ejecución (duplica!)
INSERT INTO bootcamp.silver.ventas_no_idempotente
SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id
FROM bootcamp.bronze.ventas;

SELECT COUNT(*) as filas_segunda_ejecucion FROM bootcamp.silver.ventas_no_idempotente;

-- El SUM se duplicó!
SELECT SUM(total) as total_ventas FROM bootcamp.silver.ventas_no_idempotente;

## Ejercicio 3.2: Solución 1 — INSERT OVERWRITE

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.ventas_overwrite;

CREATE TABLE bootcamp.silver.ventas_overwrite (
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;

INSERT OVERWRITE bootcamp.silver.ventas_overwrite
SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id
FROM bootcamp.bronze.ventas;

SELECT COUNT(*) as filas FROM bootcamp.silver.ventas_overwrite;

In [0]:
-- Re-ejecutar: sigue siendo 5 filas
INSERT OVERWRITE bootcamp.silver.ventas_overwrite
SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id
FROM bootcamp.bronze.ventas;

SELECT COUNT(*) as filas_re_ejecucion FROM bootcamp.silver.ventas_overwrite;

## Ejercicio 3.3: Solución 2 — MERGE

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.ventas_merge;

CREATE TABLE bootcamp.silver.ventas_merge (
    id_venta STRING NOT NULL,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;

MERGE INTO bootcamp.silver.ventas_merge AS target
USING (
    SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM bootcamp.bronze.ventas
) AS source
ON target.id_venta = source.id_venta
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT COUNT(*) as filas_primera FROM bootcamp.silver.ventas_merge;

In [0]:
-- Re-ejecutar: sigue siendo 5
MERGE INTO bootcamp.silver.ventas_merge AS target
USING (
    SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM bootcamp.bronze.ventas
) AS source
ON target.id_venta = source.id_venta
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT COUNT(*) as filas_re_ejecucion FROM bootcamp.silver.ventas_merge;

In [0]:
-- Agregar 2 nuevas ventas al bronze
INSERT INTO bootcamp.bronze.ventas VALUES
    ('V006', '2024-01-18', 'Webcam', 1, 50.00, 'C005', CURRENT_TIMESTAMP()),
    ('V007', '2024-01-18', 'Auriculares', 1, 80.00, 'C001', CURRENT_TIMESTAMP());

-- MERGE de nuevo: solo agrega las 2 nuevas
MERGE INTO bootcamp.silver.ventas_merge AS target
USING (
    SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM bootcamp.bronze.ventas
) AS source
ON target.id_venta = source.id_venta
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT COUNT(*) as filas_con_nuevas FROM bootcamp.silver.ventas_merge;

## Ejercicio 3.4: MERGE con Hash — Idempotencia simplificada

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.ventas_merge_hash;

CREATE TABLE bootcamp.silver.ventas_merge_hash (
    row_hash STRING NOT NULL,
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;

MERGE INTO bootcamp.silver.ventas_merge_hash AS target
USING (
    SELECT 
        MD5(CONCAT_WS('|', id_venta, CAST(cantidad * precio_unitario AS STRING))) AS row_hash,
        id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM bootcamp.bronze.ventas
) AS source
ON target.row_hash = source.row_hash
WHEN NOT MATCHED THEN INSERT *;

SELECT COUNT(*) as filas, COUNT(DISTINCT row_hash) as hashes_unicos
FROM bootcamp.silver.ventas_merge_hash;

In [0]:
-- Re-ejecutar: COUNT = COUNT DISTINCT (sin duplicados)
MERGE INTO bootcamp.silver.ventas_merge_hash AS target
USING (
    SELECT 
        MD5(CONCAT_WS('|', id_venta, CAST(cantidad * precio_unitario AS STRING))) AS row_hash,
        id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM bootcamp.bronze.ventas
) AS source
ON target.row_hash = source.row_hash
WHEN NOT MATCHED THEN INSERT *;

SELECT COUNT(*) as filas, COUNT(DISTINCT row_hash) as hashes_unicos
FROM bootcamp.silver.ventas_merge_hash;

## Ejercicio 3.5: INSERT OVERWRITE con particiones

In [0]:
DROP TABLE IF EXISTS bootcamp.silver.ventas_particionada;

CREATE TABLE bootcamp.silver.ventas_particionada (
    id_venta STRING,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING,
    fecha_venta DATE
) USING DELTA
PARTITIONED BY (fecha_venta);

INSERT OVERWRITE bootcamp.silver.ventas_particionada
SELECT id_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id, fecha_venta
FROM bootcamp.bronze.ventas
WHERE fecha_venta = '2024-01-15';

SELECT fecha_venta, COUNT(*) as filas
FROM bootcamp.silver.ventas_particionada
GROUP BY fecha_venta;

In [0]:
%sql
-- Re-ejecutar: la partición 2024-01-15 sigue con las mismas filas
INSERT OVERWRITE bootcamp.silver.ventas_particionada
SELECT id_venta, producto, cantidad, precio_unitario,
    cantidad * precio_unitario as total, cliente_id, fecha_venta
FROM bootcamp.bronze.ventas
WHERE fecha_venta = '2024-01-15';

SELECT fecha_venta, COUNT(*) as filas
FROM bootcamp.silver.ventas_particionada
GROUP BY fecha_venta;